# Evaluation Notebook — Sektörel Keyword Çıkartma Pipeline

**Amaç:** 30 örneklik ground-truth veri seti üzerinde pipeline metriklerini görselleştir.

Metrikler:
- Sektör Sınıflandırma: Top-1, Top-3 Accuracy, F1-Macro
- Keyword Kalitesi: Precision@K (K=3,5,10)
- Sektöre Göre Hata Analizi

In [ ]:
import json
import sys
import os
sys.path.insert(0, 'src')
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter

# Load evaluation results
with open('data/evaluation/evaluation_report.json') as f:
    report = json.load(f)

metrics = report['metrics']
details = report['details']

print('Evaluation Report Loaded')
print(f"Documents: {metrics['n_documents']}")
print(f"Top-1 Accuracy: {metrics.get('top1_accuracy',0):.1%}")
print(f"Top-3 Accuracy: {metrics.get('top3_accuracy',0):.1%}")
print(f"F1-Macro: {metrics.get('f1_macro', 'N/A')}")
print(f"Precision@3: {metrics.get('precision_at_3',0):.3f}")
print(f"Precision@5: {metrics.get('precision_at_5',0):.3f}")
print(f"Precision@10: {metrics.get('precision_at_10',0):.3f}")

## 1. Sektör Sınıflandırma Metrikleri

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Bar chart: Top-1 vs Top-3 Accuracy ---
ax = axes[0]
metric_names = ['Top-1\nAccuracy', 'Top-3\nAccuracy', 'F1-Macro']
metric_vals = [
    metrics.get('top1_accuracy', 0),
    metrics.get('top3_accuracy', 0),
    metrics.get('f1_macro', 0) or 0
]
colors = ['#2ecc71' if v >= 0.7 else '#f39c12' for v in metric_vals]
bars = ax.bar(metric_names, metric_vals, color=colors, edgecolor='white', linewidth=1.5)
ax.axhline(0.7, color='red', linestyle='--', alpha=0.5, label='Hedef (%70)')
ax.axhline(0.85, color='orange', linestyle='--', alpha=0.5, label='Üst Hedef (%85)')
for bar, val in zip(bars, metric_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_title('Sektör Sınıflandırma Metrikleri', fontweight='bold', pad=12)
ax.set_ylabel('Skor')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# --- Per-sector Top-1 performance ---
ax2 = axes[1]
df = pd.DataFrame(details)
by_sector = df.groupby('true_sector').agg(
    total=('correct_top1', 'count'),
    correct=('correct_top1', 'sum')
).reset_index()
by_sector['accuracy'] = by_sector['correct'] / by_sector['total']
by_sector = by_sector.sort_values('accuracy')
colors2 = ['#e74c3c' if v < 0.5 else '#f39c12' if v < 1.0 else '#2ecc71' for v in by_sector['accuracy']]
ax2.barh(by_sector['true_sector'], by_sector['accuracy'], color=colors2, edgecolor='white')
ax2.axvline(0.7, color='red', linestyle='--', alpha=0.5)
ax2.set_xlim(0, 1.1)
ax2.set_title('Sektöre Göre Top-1 Accuracy', fontweight='bold', pad=12)
ax2.set_xlabel('Accuracy')
for i, (_, row) in enumerate(by_sector.iterrows()):
    ax2.text(row['accuracy'] + 0.02, i, f"{row['accuracy']:.0%} ({row['correct']}/{row['total']})",
             va='center', fontsize=8)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('data/evaluation/sector_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: data/evaluation/sector_accuracy.png')

## 2. Keyword Kalite Metrikleri (Precision@K)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Precision@K bar ---
ax = axes[0]
k_vals = [3, 5, 10]
p_vals = [metrics.get(f'precision_at_{k}', 0) for k in k_vals]
colors_p = ['#3498db', '#2980b9', '#1a5276']
bars = ax.bar([f'P@{k}' for k in k_vals], p_vals, color=colors_p, edgecolor='white', linewidth=1.5)
ax.axhline(0.4, color='red', linestyle='--', alpha=0.5, label='P@3 Hedef (0.40)')
for bar, val in zip(bars, p_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_ylim(0, 0.6)
ax.set_title('Keyword Precision@K', fontweight='bold', pad=12)
ax.set_ylabel('Precision')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# --- Precision distribution per doc ---
ax2 = axes[1]
p5_vals = [d['p@5'] for d in details]
sectors = [d['true_sector'] for d in details]
colors_d = ['#2ecc71' if p >= 0.5 else '#f39c12' if p >= 0.2 else '#e74c3c' for p in p5_vals]
ax2.bar(range(len(p5_vals)), p5_vals, color=colors_d, edgecolor='white', linewidth=0.8)
ax2.set_xticks(range(len(details)))
ax2.set_xticklabels(sectors, rotation=45, ha='right', fontsize=8)
ax2.axhline(0.333, color='blue', linestyle='--', alpha=0.5, label=f'Ortalama P@5 ({sum(p5_vals)/len(p5_vals):.3f})')
ax2.set_title('Sektöre Göre P@5 (Her Doküman)', fontweight='bold', pad=12)
ax2.set_ylabel('Precision@5')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)
patch_g = mpatches.Patch(color='#2ecc71', label='P@5 ≥ 0.5 (İyi)')
patch_o = mpatches.Patch(color='#f39c12', label='P@5 0.2–0.5 (Orta)')
patch_r = mpatches.Patch(color='#e74c3c', label='P@5 < 0.2 (Zayıf)')
ax2.legend(handles=[patch_g, patch_o, patch_r], fontsize=8)

plt.tight_layout()
plt.savefig('data/evaluation/keyword_precision.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: data/evaluation/keyword_precision.png')

## 3. Hata Analizi — Yanlış Sınıflandırmalar

In [ ]:
df = pd.DataFrame(details)
errors = df[~df['correct_top1']]
print(f"Yanlış sınıflandırma: {len(errors)}/{len(df)} ({len(errors)/len(df):.1%})\n")
print('Yanlış tahminler:')
print('-' * 60)
for _, row in errors.iterrows():
    t3 = '(Top-3 doğru)' if row['correct_top3'] else '(Top-3 da yanlış)'
    print(f"  {row['legal_name'][:30]:<30} True={row['true_sector']} → Pred={row['pred_sector']} {t3}")

print('\nÖzet Tablo:')
print('='*50)
print(f"Toplam örnek         : {len(df)}")
print(f"Top-1 doğru          : {df['correct_top1'].sum()} (%{df['correct_top1'].mean()*100:.0f})")
print(f"Top-3 doğru          : {df['correct_top3'].sum()} (%{df['correct_top3'].mean()*100:.0f})")
print(f"Her ikisi de yanlış  : {(~df['correct_top3']).sum()}")
print(f"Ortalama P@5         : {df['p@5'].mean():.3f}")

## 4. Örnek Çıktı Gösterimi

In [ ]:
# Show 5 representative examples
print('ÖRNEKLEMELİ ÇIKTI GÖSTERİMİ')
print('='*70)
sample_ids = [0, 6, 11, 14, 19]  # IT, G, Q, K, D
for sid in sample_ids:
    d = details[sid]
    status = '✅' if d['correct_top1'] else '❌'
    print(f"\n{status} [{d['legal_name']}]")
    print(f"   Gerçek Sektör    : {d['true_sector']}")
    print(f"   Tahmin Sektörü   : {d['pred_sector']}")
    print(f"   Top-3 Tahminler  : {d['pred_top3']}")
    print(f"   Çıkarılan KW     : {d['extracted']}")
    print(f"   Ground Truth KW  : {d['gt']}")
    print(f"   Precision@5      : {d['p@5']}")